In [15]:
import requests
import pandas as pd
import time

In [19]:
url = "https://api.sorare.com/graphql"

team_slugs = [
    # Original Portuguese clubs
    "sporting-braga-braga",
    "sporting-cp-lisboa",
    "benfica-lisboa",
    "porto-porto",

    # Added clubs
    "besiktas-istanbul",
    "fenerbahce-istanbul",
    "galatasaray-istanbul",
    "trabzonspor-trabzon",
    "vitoria-guimaraes-guimaraes",
    "salzburg-wals-siezenheim",
    "rangers-glasgow",
    "celtic-glasgow"
]

query = """
query GetTeamPlayers($slug: String!) {
  team(slug: $slug) {
    name
    slug
    activePlayers(first: 100) {
      nodes {
        displayName
        slug
        position
      }
    }
  }
}
"""

all_players = []
failed_teams = []

for i, team_slug in enumerate(team_slugs, start=1):
    print(f"Fetching team {i}/{len(team_slugs)}: {team_slug}")

    try:
        response = requests.post(
            url,
            json={
                "query": query,
                "variables": {
                    "slug": team_slug
                }
            },
            headers={
                "Content-Type": "application/json",
                "User-Agent": "Mozilla/5.0"
            },
            timeout=30
        )

        print("Status code:", response.status_code)
        response.raise_for_status()

        data = response.json()

        if "errors" in data:
            print(f"GraphQL error for {team_slug}:")
            print(data["errors"])
            failed_teams.append(team_slug)
            continue

        team = data.get("data", {}).get("team")

        if not team:
            print(f"No team found for slug: {team_slug}")
            failed_teams.append(team_slug)
            continue

        club_name = team.get("name")
        club_slug = team.get("slug")

        players = team.get("activePlayers", {}).get("nodes", [])

        for player in players:
            all_players.append({
                "Club": club_name,
                "Club Slug": club_slug,
                "Player": player.get("displayName"),
                "Player Slug": player.get("slug"),
                "Position": player.get("position")
            })

        print(f"Found {len(players)} players for {club_name}")
        time.sleep(1)

    except Exception as e:
        print(f"Error fetching {team_slug}: {e}")
        failed_teams.append(team_slug)

print("-" * 80)

df = pd.DataFrame(all_players)

# Optional: normalize position names
position_map = {
    "Goalkeeper": "GK",
    "Defender": "DEF",
    "Midfielder": "MID",
    "Forward": "FWD",
    "goalkeeper": "GK",
    "defender": "DEF",
    "midfielder": "MID",
    "forward": "FWD"
}

df["Position"] = df["Position"].replace(position_map)

print(f"Total players extracted: {len(df)}")

if failed_teams:
    print(f"Failed teams: {len(failed_teams)}")
    for team_slug in failed_teams:
        print("-", team_slug)

display(df)

Fetching team 1/4: sporting-braga-braga
Status code: 200
Found 44 players for Sporting Braga
Fetching team 2/4: sporting-cp-lisboa
Status code: 200
Found 41 players for Sporting Clube de Portugal
Fetching team 3/4: benfica-lisboa
Status code: 200
Found 42 players for SL Benfica
Fetching team 4/4: porto-porto
Status code: 200
Found 38 players for FC Porto
--------------------------------------------------------------------------------
Total players extracted: 165


,Club,Club Slug,Player,Player Slug,Position
0,Sporting Braga,sporting-braga-braga,João Carvalho,joao-goncalo-carapinha-carvalho,GK
1,Sporting Braga,sporting-braga-braga,Jean-Baptiste Gorby,jean-baptiste-gorby,MID
2,Sporting Braga,sporting-braga-braga,Gustaf Lagerbielke,gustaf-lagerbielke,DEF
3,Sporting Braga,sporting-braga-braga,António Gil,antonio-gil-lourenco-oliveira-valerio-branco,MID
4,Sporting Braga,sporting-braga-braga,Amine El Ouazzani,amine-el-ouazzani,FWD
...,...,...,...,...,...
160,FC Porto,porto-porto,Luuk de Jong,luuk-de-jong,FWD
161,FC Porto,porto-porto,Alberto Costa,alberto-oliveira-baio,DEF
162,FC Porto,porto-porto,William,william-gomes-carvalho-santos,FWD
163,FC Porto,porto-porto,Tiago Andrade,tiago-alves-pinto-andrade,FWD


In [33]:
import requests
import pandas as pd
import json
import time
import random
import os

# ============================================================
# SETTINGS
# ============================================================

url = "https://api.sorare.com/graphql"

SEASON_START_DATE = "2022-08-01"

# For a multi-season extraction from August 2022 onward, keep Sorare's API gameWeek number.
# The old single-season website-GW offset would drop older seasons by making them <= 0.
GW_API_TO_WEBSITE_OFFSET = 0

OUTPUT_FOLDER = "sorare_outputs"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

partial_scores_file = os.path.join(OUTPUT_FOLDER, "all_players_scores_partial.csv")
completed_players_file = os.path.join(OUTPUT_FOLDER, "completed_player_slugs.json")
final_scores_file = os.path.join(OUTPUT_FOLDER, "all_players_scores_final.csv")
final_wide_file = os.path.join(OUTPUT_FOLDER, "all_players_scores_wide_final.csv")

headers = {
    "Content-Type": "application/json",
    "User-Agent": "Mozilla/5.0"
}

season_start_dt = pd.to_datetime(SEASON_START_DATE, utc=True)

# ============================================================
# API-FRIENDLY PACING
# ============================================================
# Sorare's public docs mention 20 calls/minute unauthenticated.
# This keeps normal traffic below that pace. If Sorare returns 429,
# safe_graphql_request still backs off much longer and retries.
MIN_SECONDS_BETWEEN_REQUESTS = 3.2
FIXTURE_PAGE_SLEEP_RANGE = (0.4, 0.9)
SCORE_PAGE_SLEEP_RANGE = (0.4, 0.9)
PLAYER_SLEEP_RANGE = (0.25, 0.75)
BATCH_COOLDOWN_EVERY_PLAYERS = 12
BATCH_COOLDOWN_RANGE = (6, 12)
_last_request_at = 0.0

# ============================================================
# SAFE REQUEST FUNCTION
# ============================================================

def safe_graphql_request(payload, max_retries=8):
    global _last_request_at

    for attempt in range(1, max_retries + 1):
        elapsed = time.monotonic() - _last_request_at
        wait_time = max(0, MIN_SECONDS_BETWEEN_REQUESTS - elapsed)
        wait_time += random.uniform(0.05, 0.25)
        if wait_time > 0:
            time.sleep(wait_time)

        response = requests.post(
            url,
            json=payload,
            headers=headers,
            timeout=30
        )
        _last_request_at = time.monotonic()

        print(f"Status code: {response.status_code}")

        if response.status_code == 422:
            print("422 Unprocessable Entity. GraphQL query is invalid.")
            try:
                print(json.dumps(response.json(), indent=2))
            except Exception:
                print(response.text)
            raise Exception("Stopping because Sorare rejected the GraphQL query.")

        if response.status_code == 429:
            retry_after = response.headers.get("Retry-After")
            if retry_after:
                wait_time = int(retry_after)
            else:
                wait_time = min(300, 20 * attempt) + random.uniform(10, 30)
            print(f"429 Too Many Requests. Waiting {wait_time:.1f} seconds...")
            time.sleep(wait_time)
            continue

        if response.status_code in [500, 502, 503, 504]:
            wait_time = min(300, 15 * attempt) + random.uniform(10, 30)
            print(f"Temporary server error {response.status_code}. Waiting {wait_time:.1f} seconds...")
            time.sleep(wait_time)
            continue

        response.raise_for_status()
        data = response.json()

        if "errors" in data:
            print("GraphQL errors returned:")
            print(json.dumps(data["errors"], indent=2))
            raise Exception("Stopping because GraphQL returned errors.")

        return data

    raise Exception("Max retries reached.")


# ============================================================
# 1. FETCH SO5 FIXTURE CALENDAR
# Stop when fixture coverage reaches SEASON_START_DATE.
# ============================================================

fixtures_query = """
query GetSo5Fixtures($after: String) {
  so5 {
    so5Fixtures(first: 40, after: $after) {
      nodes {
        gameWeek
        startDate
        endDate
      }
      pageInfo {
        hasNextPage
        endCursor
      }
    }
  }
}
"""

print("=" * 100)
print("Fetching Sorare SO5 fixture calendar...")
print("=" * 100)

all_fixtures = []
after_cursor = None
page_number = 0

while True:
    page_number += 1
    print("-" * 100)
    print(f"Fetching fixture page {page_number}")

    data_fixtures_page = safe_graphql_request({
        "query": fixtures_query,
        "variables": {"after": after_cursor}
    })

    connection = (
        data_fixtures_page
        .get("data", {})
        .get("so5", {})
        .get("so5Fixtures", {})
    )

    nodes = connection.get("nodes", [])
    page_info = connection.get("pageInfo", {})
    print(f"Fixtures in this page: {len(nodes)}")

    all_fixtures.extend(nodes)
    df_temp_fixtures = pd.DataFrame(all_fixtures)

    if not df_temp_fixtures.empty:
        df_temp_fixtures["startDate"] = pd.to_datetime(df_temp_fixtures["startDate"], utc=True, errors="coerce")
        df_temp_fixtures["endDate"] = pd.to_datetime(df_temp_fixtures["endDate"], utc=True, errors="coerce")
        min_fixture_start = df_temp_fixtures["startDate"].min()
        max_fixture_end = df_temp_fixtures["endDate"].max()
        print(f"Current fixture coverage: {min_fixture_start} to {max_fixture_end}")

        if min_fixture_start <= season_start_dt:
            print(f"Fixture calendar now covers {SEASON_START_DATE}. Stopping fixture pagination.")
            break

    has_next_page = page_info.get("hasNextPage")
    after_cursor = page_info.get("endCursor")

    if not has_next_page:
        print("No more fixture pages available.")
        break

    sleep_time = random.uniform(*FIXTURE_PAGE_SLEEP_RANGE)
    print(f"Sleeping {sleep_time:.1f} seconds before next fixture page...")
    time.sleep(sleep_time)


df_fixtures = pd.DataFrame(all_fixtures)

if df_fixtures.empty:
    raise Exception("No fixtures returned. Cannot map dates to GWs.")

df_fixtures["startDate"] = pd.to_datetime(df_fixtures["startDate"], utc=True, errors="coerce")
df_fixtures["endDate"] = pd.to_datetime(df_fixtures["endDate"], utc=True, errors="coerce")
df_fixtures["gameWeek"] = pd.to_numeric(df_fixtures["gameWeek"], errors="coerce")

df_fixtures = (
    df_fixtures
    .dropna(subset=["startDate", "endDate", "gameWeek"])
    .drop_duplicates()
    .sort_values("gameWeek", ascending=False)
    .reset_index(drop=True)
)

df_fixtures["Sorare GW API"] = df_fixtures["gameWeek"].astype(int)
df_fixtures["Sorare GW"] = df_fixtures["Sorare GW API"] - GW_API_TO_WEBSITE_OFFSET

df_fixtures = df_fixtures[df_fixtures["Sorare GW"] >= 1].copy()

print("=" * 100)
print("Fixture calendar fetched.")
print(f"Total relevant fixtures fetched: {len(df_fixtures)}")
print(f"Relevant fixture date coverage: {df_fixtures['startDate'].min()} to {df_fixtures['endDate'].max()}")
print(f"Sorare API GW range fetched: {int(df_fixtures['Sorare GW'].min())} to {int(df_fixtures['Sorare GW'].max())}")
print("=" * 100)

display(df_fixtures.head(20))


# ============================================================
# 2. HELPER TO MAP GAME DATE TO SO5 GAME WEEK
# ============================================================

def assign_sorare_gw_api(game_date, fixtures_df):
    if pd.isna(game_date):
        return None

    match = fixtures_df[
        (fixtures_df["startDate"] <= game_date) &
        (fixtures_df["endDate"] >= game_date)
    ]

    if match.empty:
        return None

    return int(match.iloc[0]["Sorare GW API"])


# ============================================================
# 3. FETCH WHOLE-HISTORY SCORES FOR EVERY PLAYER
# ============================================================
# IMPORTANT:
# The selected club slugs are used to build the player list/current squad context.
# allSo5Scores returns the player's Sorare score history since SEASON_START_DATE,
# not only scores while playing for the selected/current squad club.

player_scores_query = """
query GetPlayerScores($slug: String!, $after: String) {
  football {
    player(slug: $slug) {
      displayName
      slug
      position
      allSo5Scores(first: 40, after: $after) {
        nodes {
          score
          game {
            date
          }
        }
        pageInfo {
          hasNextPage
          endCursor
        }
      }
    }
  }
}
"""

player_rows = (
    df[["Club", "Club Slug", "Player", "Player Slug", "Position"]]
    .dropna(subset=["Player Slug"])
    .drop_duplicates()
    .reset_index(drop=True)
)

total_players = len(player_rows)

# ============================================================
# RESUME / CHECKPOINT SUPPORT
# ============================================================
# If the notebook stops halfway, rerun this cell and it will skip players
# that were already fully extracted.
if os.path.exists(partial_scores_file):
    df_partial_existing = pd.read_csv(partial_scores_file)
    all_score_rows = df_partial_existing.to_dict("records")
    print(f"Loaded partial score checkpoint: {len(all_score_rows)} rows")
else:
    all_score_rows = []

if os.path.exists(completed_players_file):
    with open(completed_players_file, "r", encoding="utf-8") as f:
        completed_player_slugs = set(json.load(f))
    print(f"Loaded completed player checkpoint: {len(completed_player_slugs)} players")
else:
    completed_player_slugs = set()

player_rows_to_fetch = (
    player_rows[~player_rows["Player Slug"].isin(completed_player_slugs)]
    .reset_index(drop=True)
)
players_to_fetch = len(player_rows_to_fetch)

print("=" * 100)
print(f"Starting whole-history score extraction for {players_to_fetch}/{total_players} players")
print(f"Start date: {SEASON_START_DATE}")
print("=" * 100)

for player_index, player_row in player_rows_to_fetch.iterrows():
    club = player_row["Club"]
    club_slug = player_row["Club Slug"]
    player_name_from_df = player_row["Player"]
    player_slug = player_row["Player Slug"]
    position_from_df = player_row["Position"]

    print("-" * 100)
    print(f"Fetching player {player_index + 1}/{players_to_fetch}")
    print(f"Selected/current squad club: {club}")
    print(f"Player: {player_name_from_df}")
    print(f"Position: {position_from_df}")
    print(f"Player Slug: {player_slug}")

    after_cursor = None
    score_page_number = 0
    player_score_rows = []
    player_info = None

    while True:
        score_page_number += 1
        print(f"Fetching score page {score_page_number} for {player_name_from_df}")

        data_page = safe_graphql_request({
            "query": player_scores_query,
            "variables": {
                "slug": player_slug,
                "after": after_cursor
            }
        })

        player = (
            data_page
            .get("data", {})
            .get("football", {})
            .get("player")
        )

        if not player:
            print(f"No player object returned for {player_slug}. Retrying after wait...")
            time.sleep(random.uniform(20, 40))
            continue

        if player_info is None:
            player_info = {
                "Player": player.get("displayName") or player_name_from_df,
                "Player Slug": player.get("slug") or player_slug,
                "Position": player.get("position") or position_from_df
            }

        connection = player.get("allSo5Scores", {})
        nodes = connection.get("nodes", [])
        page_info = connection.get("pageInfo", {})
        print(f"Scores in this page: {len(nodes)}")

        page_dates = []

        for item in nodes:
            game = item.get("game") or {}
            score = item.get("score")
            game_date = game.get("date")

            if not game_date:
                continue

            game_dt = pd.to_datetime(game_date, utc=True, errors="coerce")
            if pd.isna(game_dt):
                continue

            page_dates.append(game_dt)

            if game_dt >= season_start_dt:
                player_score_rows.append({
                    "Club": club,
                    "Club Slug": club_slug,
                    "Player": player_info["Player"],
                    "Player Slug": player_info["Player Slug"],
                    "Position": player_info["Position"],
                    "Score": round(float(score), 1) if score is not None else None,
                    "Game Date": game_dt,
                    "Selected Club": club,
                    "Selected Club Slug": club_slug,
                    "Score Scope": "Whole player history since " + SEASON_START_DATE
                })

        if page_dates:
            oldest_date_in_page = min(page_dates)
            print(f"Oldest score date in this page: {oldest_date_in_page}")

            if oldest_date_in_page < season_start_dt:
                print(f"Reached scores before {SEASON_START_DATE}. Stopping pagination for this player.")
                break

        has_next_page = page_info.get("hasNextPage")
        after_cursor = page_info.get("endCursor")

        if not has_next_page:
            print("No more score pages for this player.")
            break

        sleep_time = random.uniform(*SCORE_PAGE_SLEEP_RANGE)
        print(f"Sleeping {sleep_time:.1f} seconds before next score page...")
        time.sleep(sleep_time)

    if player_score_rows:
        df_player_scores_temp = pd.DataFrame(player_score_rows)
        df_player_scores_temp["Sorare GW API"] = df_player_scores_temp["Game Date"].apply(
            lambda x: assign_sorare_gw_api(x, df_fixtures)
        )
        df_player_scores_temp["Sorare GW"] = df_player_scores_temp["Sorare GW API"] - GW_API_TO_WEBSITE_OFFSET
        df_player_scores_temp = df_player_scores_temp[df_player_scores_temp["Sorare GW"].notna()].copy()
        player_score_rows = df_player_scores_temp.to_dict("records")

    all_score_rows.extend(player_score_rows)
    print(f"Kept score rows for this player: {len(player_score_rows)}")

    completed_player_slugs.add(player_slug)

    if all_score_rows:
        pd.DataFrame(all_score_rows).to_csv(partial_scores_file, index=False)
    with open(completed_players_file, "w", encoding="utf-8") as f:
        json.dump(sorted(completed_player_slugs), f, indent=2)
    print(f"Progress saved to: {partial_scores_file}")
    print(f"Completed players saved to: {completed_players_file}")

    sleep_time = random.uniform(*PLAYER_SLEEP_RANGE)
    print(f"Short pause {sleep_time:.1f} seconds before next player...")
    time.sleep(sleep_time)

    players_done = player_index + 1
    if players_done % BATCH_COOLDOWN_EVERY_PLAYERS == 0 and players_done < players_to_fetch:
        cooldown = random.uniform(*BATCH_COOLDOWN_RANGE)
        print(f"Batch cooldown after {players_done} players: {cooldown:.1f} seconds...")
        time.sleep(cooldown)


# ============================================================
# 4. FINAL LONG SCORE DATAFRAME
# ============================================================

print("=" * 100)
print("Finished extracting scores for all players.")
print("=" * 100)

df_scores = pd.DataFrame(all_score_rows)

if not df_scores.empty:
    df_scores["Game Date"] = pd.to_datetime(df_scores["Game Date"], utc=True, errors="coerce")
    df_scores["Sorare GW API"] = pd.to_numeric(df_scores["Sorare GW API"], errors="coerce")
    df_scores["Sorare GW"] = pd.to_numeric(df_scores["Sorare GW"], errors="coerce")
    df_scores["Score"] = pd.to_numeric(df_scores["Score"], errors="coerce").round(1)

    df_scores = (
        df_scores
        .dropna(subset=["Sorare GW"])
        .drop_duplicates(subset=["Player Slug", "Game Date", "Score"], keep="last")
        .sort_values(["Sorare GW", "Club", "Player"], ascending=[False, True, True])
        .reset_index(drop=True)
    )

df_scores.to_csv(final_scores_file, index=False)

print(f"Total score rows extracted: {len(df_scores)}")
print(f"Saved long score table to: {final_scores_file}")
print("Note: these are whole player-history scores from players found in the selected/current squad clubs.")

display(df_scores)


# ============================================================
# 5. CREATE FULL PLAYER x GW GRID
# Includes 0 for missing / benched / DNP gameweeks.
# ============================================================

print("=" * 100)
print("Creating full Player x GW table...")
print("=" * 100)

if df_scores.empty:
    raise Exception("df_scores is empty. Cannot create wide table.")

max_gw = int(df_scores["Sorare GW"].max())
min_gw = int(df_scores["Sorare GW"].min())

all_gws = pd.DataFrame({
    "Sorare GW": range(max_gw, min_gw - 1, -1)
})

players = (
    df[["Club", "Club Slug", "Player", "Player Slug", "Position"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

full_grid = players.merge(all_gws, how="cross")

df_scores_for_merge = df_scores[
    [
        "Club",
        "Club Slug",
        "Player",
        "Player Slug",
        "Position",
        "Sorare GW",
        "Score"
    ]
].copy()

df_scores_full = full_grid.merge(
    df_scores_for_merge,
    on=["Club", "Club Slug", "Player", "Player Slug", "Position", "Sorare GW"],
    how="left"
)

df_scores_full["Score Status"] = df_scores_full["Score"].apply(
    lambda x: "No Score / Benched / DNP" if pd.isna(x) else "Played / Has Score"
)

df_scores_full["Score"] = df_scores_full["Score"].fillna(0).round(1)
df_scores_full["GW Column"] = "GW " + df_scores_full["Sorare GW"].astype(int).astype(str)

df_players_gw = df_scores_full.pivot_table(
    index=["Club", "Club Slug", "Player", "Player Slug", "Position"],
    columns="GW Column",
    values="Score",
    aggfunc="first"
).reset_index()

id_cols = ["Club", "Club Slug", "Player", "Player Slug", "Position"]

gw_cols = sorted(
    [col for col in df_players_gw.columns if col.startswith("GW ")],
    key=lambda x: int(x.replace("GW ", "")),
    reverse=True
)

df_players_gw = df_players_gw[id_cols + gw_cols]

for col in gw_cols:
    df_players_gw[col] = df_players_gw[col].round(1)

df_players_gw.to_csv(final_wide_file, index=False)

print(f"Wide table created: {len(df_players_gw)} players")
print(f"GW columns: GW {max_gw} down to GW {min_gw}")
print(f"Saved wide score table to: {final_wide_file}")

display(df_players_gw)


Fetching Sorare SO5 fixture calendar...
----------------------------------------------------------------------------------------------------
Fetching fixture page 1
Status code: 200
Fixtures in this page: 40
Current fixture coverage: 2026-01-06 15:00:00+00:00 to 2026-05-26 13:59:00+00:00
Sleeping 3.5 seconds before next fixture page...
----------------------------------------------------------------------------------------------------
Fetching fixture page 2
Status code: 200
Fixtures in this page: 40
Current fixture coverage: 2025-08-19 21:00:00+00:00 to 2026-05-26 13:59:00+00:00
Sleeping 3.3 seconds before next fixture page...
----------------------------------------------------------------------------------------------------
Fetching fixture page 3
Status code: 200
Fixtures in this page: 40
Current fixture coverage: 2025-03-25 15:00:00+00:00 to 2026-05-26 13:59:00+00:00
Fixture calendar now covers 2025-08-01. Stopping fixture pagination.
Fixture calendar fetched.
Total relevant fixtu

,gameWeek,startDate,endDate,Sorare GW API,Sorare GW
0,685,2026-05-22 14:00:00+00:00,2026-05-26 13:59:00+00:00,685,81
1,684,2026-05-19 14:00:00+00:00,2026-05-22 13:59:00+00:00,684,80
2,683,2026-05-15 14:00:00+00:00,2026-05-19 13:59:00+00:00,683,79
3,682,2026-05-12 14:00:00+00:00,2026-05-15 13:59:00+00:00,682,78
4,681,2026-05-08 14:00:00+00:00,2026-05-12 13:59:00+00:00,681,77
5,680,2026-05-05 14:00:00+00:00,2026-05-08 13:59:00+00:00,680,76
6,679,2026-05-01 14:00:00+00:00,2026-05-05 13:59:00+00:00,679,75
7,678,2026-04-28 14:00:00+00:00,2026-05-01 13:59:00+00:00,678,74
8,677,2026-04-24 14:00:00+00:00,2026-04-28 13:59:00+00:00,677,73
9,676,2026-04-21 14:00:00+00:00,2026-04-24 13:59:00+00:00,676,72


Starting score extraction for 165 players
Season start date: 2025-08-01
----------------------------------------------------------------------------------------------------
Fetching player 1/165
Club: Sporting Braga
Player: João Carvalho
Position: GK
Player Slug: joao-goncalo-carapinha-carvalho
Fetching score page 1 for João Carvalho
Status code: 200
Scores in this page: 40
Oldest score date in this page: 2025-10-05 18:15:00+00:00
Sleeping 2.5 seconds before next score page...
Fetching score page 2 for João Carvalho
Status code: 200
Scores in this page: 34
Oldest score date in this page: 2025-02-03 18:45:00+00:00
Reached scores before 2025-08-01. Stopping pagination for this player.
Kept score rows for this player: 50
Progress saved to: sorare_outputs\all_players_scores_partial.csv
Sleeping 8.9 seconds before next player...
----------------------------------------------------------------------------------------------------
Fetching player 2/165
Club: Sporting Braga
Player: Jean-Baptist

,Club,Club Slug,Player,Player Slug,Position,Score,Game Date,Sorare GW API,Sorare GW
0,FC Porto,porto-porto,Alan Varela,alan-varela,Midfielder,39.7,2026-05-10 17:00:00+00:00,681.0,77.0
1,FC Porto,porto-porto,Alberto Costa,alberto-oliveira-baio,Defender,48.5,2026-05-10 17:00:00+00:00,681.0,77.0
2,FC Porto,porto-porto,André Miranda,andre-alexandre-mendes-miranda,Forward,0.0,2026-05-10 17:00:00+00:00,681.0,77.0
3,FC Porto,porto-porto,André Oliveira,andre-luis-rocha-oliveira,Midfielder,0.0,2026-05-10 17:00:00+00:00,681.0,77.0
4,FC Porto,porto-porto,Borja Sainz,borja-sainz-eguskiza,Midfielder,60.0,2026-05-10 17:00:00+00:00,681.0,77.0
...,...,...,...,...,...,...,...,...,...
6572,Sporting Clube de Portugal,sporting-cp-lisboa,Ricardo Mangas,ricardo-luis-chaby-mangas,Defender,100.0,2025-08-17 19:30:00+00:00,605.0,1.0
6573,Sporting Clube de Portugal,sporting-cp-lisboa,Rui Silva,rui-tiago-dantas-da-silva,Goalkeeper,71.0,2025-08-17 19:30:00+00:00,605.0,1.0
6574,Sporting Clube de Portugal,sporting-cp-lisboa,Souleymane Faye,souleymane-faye,Forward,0.0,2025-08-16 19:30:00+00:00,605.0,1.0
6575,Sporting Clube de Portugal,sporting-cp-lisboa,Trincão,francisco-antonio-machado-mota-de-castro-trincao,Forward,82.6,2025-08-17 19:30:00+00:00,605.0,1.0


Creating full Player x GW table...
Wide table created: 165 players
GW columns: GW 77 down to GW 1
Saved wide score table to: sorare_outputs\all_players_scores_wide_final.csv


GW Column,Club,Club Slug,Player,Player Slug,Position,GW 77,GW 76,GW 75,GW 74,GW 73,...,GW 10,GW 9,GW 8,GW 7,GW 6,GW 5,GW 4,GW 3,GW 2,GW 1
0,FC Porto,porto-porto,Alan Varela,alan-varela,MID,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,FC Porto,porto-porto,Alberto Costa,alberto-oliveira-baio,DEF,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,FC Porto,porto-porto,André Miranda,andre-alexandre-mendes-miranda,FWD,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,FC Porto,porto-porto,André Oliveira,andre-luis-rocha-oliveira,MID,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,FC Porto,porto-porto,Borja Sainz,borja-sainz-eguskiza,MID,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
160,Sporting Clube de Portugal,sporting-cp-lisboa,Salvador Blopa,salvador-blopa,DEF,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
161,Sporting Clube de Portugal,sporting-cp-lisboa,Samuel Justo,samuel-loureiro-carvalho-justo,MID,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
162,Sporting Clube de Portugal,sporting-cp-lisboa,Souleymane Faye,souleymane-faye,FWD,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
163,Sporting Clube de Portugal,sporting-cp-lisboa,Trincão,francisco-antonio-machado-mota-de-castro-trincao,FWD,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Let's check the attributes that were returned by checking the `.keys()` of our dict.

In [39]:
import pandas as pd
import os

OUTPUT_FOLDER = "sorare_outputs"
final_wide_file = os.path.join(OUTPUT_FOLDER, "all_players_scores_wide_final_fixed.csv")

# ============================================================
# 1. CLEAN PLAYERS TABLE
# ============================================================

players = (
    df[["Club", "Club Slug", "Player", "Player Slug", "Position"]]
    .dropna(subset=["Player Slug"])
    .drop_duplicates(subset=["Player Slug"])
    .copy()
)

# Remove coaches
players = players[players["Position"] != "Coach"].copy()

# ============================================================
# 2. CLEAN SCORES TABLE
# ============================================================

df_scores_clean = df_scores.copy()

df_scores_clean["Sorare GW"] = pd.to_numeric(
    df_scores_clean["Sorare GW"],
    errors="coerce"
)

df_scores_clean["Score"] = pd.to_numeric(
    df_scores_clean["Score"],
    errors="coerce"
).round(1)

df_scores_clean = df_scores_clean.dropna(subset=["Player Slug", "Sorare GW"])

df_scores_clean["Sorare GW"] = df_scores_clean["Sorare GW"].astype(int)

# Keep only GW >= 1
df_scores_clean = df_scores_clean[df_scores_clean["Sorare GW"] >= 1].copy()

# IMPORTANT:
# Use only Player Slug + Sorare GW for merging.
# Do not merge on Player name, Club, or Position.
df_scores_for_merge = df_scores_clean[
    ["Player Slug", "Sorare GW", "Score"]
].copy()

# If duplicate score rows exist for same player/GW, keep the highest score
df_scores_for_merge = (
    df_scores_for_merge
    .groupby(["Player Slug", "Sorare GW"], as_index=False)["Score"]
    .max()
)

# ============================================================
# 3. CREATE FULL PLAYER x GW GRID
# ============================================================

max_gw = int(df_scores_clean["Sorare GW"].max())
min_gw = 1

all_gws = pd.DataFrame({
    "Sorare GW": range(max_gw, min_gw - 1, -1)
})

full_grid = players.merge(all_gws, how="cross")

df_scores_full = full_grid.merge(
    df_scores_for_merge,
    on=["Player Slug", "Sorare GW"],
    how="left"
)

df_scores_full["Score Status"] = df_scores_full["Score"].apply(
    lambda x: "No Score / Benched / DNP" if pd.isna(x) else "Played / Has Score"
)

df_scores_full["Score"] = df_scores_full["Score"].fillna(0).round(1)

# ============================================================
# 4. CREATE WIDE TABLE
# ============================================================

df_scores_full["GW Column"] = "GW " + df_scores_full["Sorare GW"].astype(str)

df_players_gw = df_scores_full.pivot_table(
    index=["Club", "Club Slug", "Player", "Player Slug", "Position"],
    columns="GW Column",
    values="Score",
    aggfunc="first"
).reset_index()

id_cols = ["Club", "Club Slug", "Player", "Player Slug", "Position"]

gw_cols = sorted(
    [col for col in df_players_gw.columns if col.startswith("GW ")],
    key=lambda x: int(x.replace("GW ", "")),
    reverse=True
)

df_players_gw = df_players_gw[id_cols + gw_cols]

for col in gw_cols:
    df_players_gw[col] = df_players_gw[col].round(1)

df_players_gw.to_csv(final_wide_file, index=False)

print("=" * 100)
print("Fixed wide table created.")
print(f"Players: {len(df_players_gw)}")
print(f"GW columns: GW {max_gw} down to GW 1")
print(f"Saved to: {final_wide_file}")
print("=" * 100)

display(df_players_gw)

Fixed wide table created.
Players: 161
GW columns: GW 77 down to GW 1
Saved to: sorare_outputs\all_players_scores_wide_final_fixed.csv


GW Column,Club,Club Slug,Player,Player Slug,Position,GW 77,GW 76,GW 75,GW 74,GW 73,...,GW 10,GW 9,GW 8,GW 7,GW 6,GW 5,GW 4,GW 3,GW 2,GW 1
0,FC Porto,porto-porto,Alan Varela,alan-varela,MID,39.7,0.0,52.2,0.0,0.0,...,0.0,39.2,0.0,0.0,0.0,49.2,0.0,53.0,0.0,34.2
1,FC Porto,porto-porto,Alberto Costa,alberto-oliveira-baio,DEF,48.5,0.0,66.0,0.0,78.9,...,0.0,0.0,0.0,0.0,0.0,60.0,0.0,89.2,0.0,44.4
2,FC Porto,porto-porto,André Miranda,andre-alexandre-mendes-miranda,FWD,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,FC Porto,porto-porto,André Oliveira,andre-luis-rocha-oliveira,MID,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,FC Porto,porto-porto,Borja Sainz,borja-sainz-eguskiza,MID,60.0,0.0,34.9,0.0,0.0,...,0.0,74.8,0.0,0.0,0.0,33.5,0.0,79.0,0.0,32.3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
156,Sporting Clube de Portugal,sporting-cp-lisboa,Salvador Blopa,salvador-blopa,DEF,0.0,0.0,0.0,30.1,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
157,Sporting Clube de Portugal,sporting-cp-lisboa,Samuel Justo,samuel-loureiro-carvalho-justo,MID,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
158,Sporting Clube de Portugal,sporting-cp-lisboa,Souleymane Faye,souleymane-faye,FWD,0.0,0.0,0.0,0.0,0.0,...,0.0,29.8,0.0,39.1,0.0,45.6,0.0,41.6,0.0,0.0
159,Sporting Clube de Portugal,sporting-cp-lisboa,Trincão,francisco-antonio-machado-mota-de-castro-trincao,FWD,75.8,0.0,35.8,43.7,48.3,...,93.6,46.6,35.8,37.7,0.0,40.8,0.0,41.1,0.0,82.6


In [40]:
import pandas as pd
import os

# ============================================================
# SETTINGS
# ============================================================

OUTPUT_FOLDER = "sorare_outputs"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

top_lineups_file = os.path.join(OUTPUT_FOLDER, "top_4_lineups_per_gw_clean.csv")

VALID_POSITIONS = ["GK", "DEF", "MID", "FWD"]

# ============================================================
# CLEAN DATA
# ============================================================

df_lineup_source = df_players_gw.copy()

df_lineup_source = df_lineup_source[
    df_lineup_source["Position"].isin(VALID_POSITIONS)
].copy()

gw_cols = sorted(
    [col for col in df_lineup_source.columns if col.startswith("GW ")],
    key=lambda x: int(x.replace("GW ", "")),
    reverse=True
)

print(f"GW columns found: {len(gw_cols)}")
print(gw_cols[:10])


# ============================================================
# FAST LINEUP BUILDER
# ============================================================

def get_top_players_by_position(gw_df, position, n):
    pos_df = gw_df[gw_df["Position"] == position].copy()

    pos_df = pos_df.sort_values(
        ["GW Score", "Player"],
        ascending=[False, True]
    )

    return pos_df.head(n)


def build_best_team_fast(available_players, gw_col):
    gw_df = available_players.copy()

    gw_df["GW Score"] = pd.to_numeric(
        gw_df[gw_col],
        errors="coerce"
    ).fillna(0).round(1)

    formations = [
        {
            "Formation": "Extra DEF",
            "Needs": {"GK": 1, "DEF": 2, "MID": 1, "FWD": 1},
            "Extra Position": "DEF"
        },
        {
            "Formation": "Extra MID",
            "Needs": {"GK": 1, "DEF": 1, "MID": 2, "FWD": 1},
            "Extra Position": "MID"
        },
        {
            "Formation": "Extra FWD",
            "Needs": {"GK": 1, "DEF": 1, "MID": 1, "FWD": 2},
            "Extra Position": "FWD"
        }
    ]

    candidate_teams = []

    for formation in formations:
        selected_parts = []

        possible = True

        for position, n_needed in formation["Needs"].items():
            top_players = get_top_players_by_position(gw_df, position, n_needed)

            if len(top_players) < n_needed:
                possible = False
                break

            selected_parts.append(top_players)

        if not possible:
            continue

        selected = pd.concat(selected_parts, ignore_index=True)

        if selected["Player Slug"].duplicated().any():
            continue

        total_score = round(selected["GW Score"].sum(), 1)

        candidate_teams.append({
            "Formation": formation["Formation"],
            "Extra Position": formation["Extra Position"],
            "Total Score": total_score,
            "Selected": selected
        })

    if not candidate_teams:
        return None

    best_team = sorted(
        candidate_teams,
        key=lambda x: x["Total Score"],
        reverse=True
    )[0]

    return best_team


# ============================================================
# BUILD TOP 4 TEAMS PER GW
# ============================================================

all_lineup_rows = []

for gw_col in gw_cols:
    print("=" * 100)
    print(f"Building top 4 teams for {gw_col}")
    print("=" * 100)

    available_players = df_lineup_source.copy()

    for team_rank in range(1, 5):
        best_team = build_best_team_fast(available_players, gw_col)

        if best_team is None:
            print(f"Could not build team rank {team_rank} for {gw_col}")
            break

        selected = best_team["Selected"].copy()
        extra_position = best_team["Extra Position"]

        # Sort each selected group by score, so when there are 2 DEF/MID/FWD,
        # the better one becomes the normal position and the second becomes Extra.
        selected = selected.sort_values(
            ["Position", "GW Score"],
            ascending=[True, False]
        )

        total_score = round(selected["GW Score"].sum(), 1)

        row = {
            "GW": gw_col,
            "Team Rank": team_rank,
            "Formation": best_team["Formation"],
            "Total Score": total_score
        }

        # Assign one standard player per required position
        for position in ["GK", "DEF", "MID", "FWD"]:
            pos_players = selected[selected["Position"] == position].copy()

            if pos_players.empty:
                row[f"{position} Player"] = None
                row[f"{position} Club"] = None
                row[f"{position} Score"] = None
                continue

            # Highest score in this position becomes the main position player
            main_player = pos_players.sort_values(
                ["GW Score", "Player"],
                ascending=[False, True]
            ).iloc[0]

            row[f"{position} Player"] = main_player["Player"]
            row[f"{position} Club"] = main_player["Club"]
            row[f"{position} Score"] = round(float(main_player["GW Score"]), 1)

        # Assign Extra player
        extra_candidates = selected[selected["Position"] == extra_position].copy()

        # Extra is the second player in the extra position group
        extra_candidates = extra_candidates.sort_values(
            ["GW Score", "Player"],
            ascending=[False, True]
        )

        if len(extra_candidates) >= 2:
            extra_player = extra_candidates.iloc[1]
        else:
            # Safety fallback, should not happen with valid formations
            extra_pool = selected[
                selected["Position"].isin(["DEF", "MID", "FWD"])
            ].sort_values(
                ["GW Score", "Player"],
                ascending=[False, True]
            )
            extra_player = extra_pool.iloc[-1]

        row["Extra Player"] = extra_player["Player"]
        row["Extra Club"] = extra_player["Club"]
        row["Extra Position"] = extra_player["Position"]
        row["Extra Score"] = round(float(extra_player["GW Score"]), 1)

        all_lineup_rows.append(row)

        print(f"{gw_col} | Team {team_rank} | {best_team['Formation']} | Total Score: {total_score}")
        print(f"  GK: {row['GK Player']} | {row['GK Club']} | {row['GK Score']}")
        print(f"  DEF: {row['DEF Player']} | {row['DEF Club']} | {row['DEF Score']}")
        print(f"  MID: {row['MID Player']} | {row['MID Club']} | {row['MID Score']}")
        print(f"  FWD: {row['FWD Player']} | {row['FWD Club']} | {row['FWD Score']}")
        print(
            f"  Extra: {row['Extra Player']} | {row['Extra Club']} | "
            f"{row['Extra Position']} | {row['Extra Score']}"
        )

        # Remove selected players for the next team
        selected_slugs = selected["Player Slug"].tolist()

        available_players = available_players[
            ~available_players["Player Slug"].isin(selected_slugs)
        ].copy()

        print("-" * 100)


# ============================================================
# FINAL CLEAN DATAFRAME
# ============================================================

df_top_4_lineups = pd.DataFrame(all_lineup_rows)

ordered_cols = [
    "GW",
    "Team Rank",
    "Formation",
    "Total Score",
    "GK Player",
    "GK Club",
    "GK Score",
    "DEF Player",
    "DEF Club",
    "DEF Score",
    "MID Player",
    "MID Club",
    "MID Score",
    "FWD Player",
    "FWD Club",
    "FWD Score",
    "Extra Player",
    "Extra Club",
    "Extra Position",
    "Extra Score"
]

df_top_4_lineups = df_top_4_lineups[ordered_cols]

df_top_4_lineups.to_csv(top_lineups_file, index=False)

print("=" * 100)
print("Finished building clean top 4 lineups per GW.")
print(f"Saved to: {top_lineups_file}")
print("=" * 100)

display(df_top_4_lineups)

GW columns found: 77
['GW 77', 'GW 76', 'GW 75', 'GW 74', 'GW 73', 'GW 72', 'GW 71', 'GW 70', 'GW 69', 'GW 68']
Building top 4 teams for GW 77
GW 77 | Team 1 | Extra FWD | Total Score: 363.5
  GK: Lukáš Horníček | Sporting Braga | 48.4
  DEF: António Silva | SL Benfica | 74.2
  MID: Rafa | SL Benfica | 79.1
  FWD: Andreas Schjelderup | SL Benfica | 83.6
  Extra: Pau Víctor | Sporting Braga | FWD | 78.2
----------------------------------------------------------------------------------------------------
GW 77 | Team 2 | Extra FWD | Total Score: 325.0
  GK: Rui Silva | Sporting Clube de Portugal | 43.3
  DEF: Ousmane Diomande | Sporting Clube de Portugal | 64.6
  MID: Maho Dorgeles | Sporting Braga | 69.6
  FWD: Trincão | Sporting Clube de Portugal | 75.8
  Extra: Vangelis Pavlidis | SL Benfica | FWD | 71.7
----------------------------------------------------------------------------------------------------
GW 77 | Team 3 | Extra MID | Total Score: 287.4
  GK: Anatolii Trubin | SL Benfica 

,GW,Team Rank,Formation,Total Score,GK Player,GK Club,GK Score,DEF Player,DEF Club,DEF Score,MID Player,MID Club,MID Score,FWD Player,FWD Club,FWD Score,Extra Player,Extra Club,Extra Position,Extra Score
0,GW 77,1,Extra FWD,363.5,Lukáš Horníček,Sporting Braga,48.4,António Silva,SL Benfica,74.2,Rafa,SL Benfica,79.1,Andreas Schjelderup,SL Benfica,83.6,Pau Víctor,Sporting Braga,FWD,78.2
1,GW 77,2,Extra FWD,325.0,Rui Silva,Sporting Clube de Portugal,43.3,Ousmane Diomande,Sporting Clube de Portugal,64.6,Maho Dorgeles,Sporting Braga,69.6,Trincão,Sporting Clube de Portugal,75.8,Vangelis Pavlidis,SL Benfica,FWD,71.7
2,GW 77,3,Extra MID,287.4,Anatolii Trubin,SL Benfica,25.5,Víctor Gómez,Sporting Braga,60.0,Fredrik Aursnes,SL Benfica,67.5,Luis Suárez,Sporting Clube de Portugal,67.7,Jean-Baptiste Gorby,Sporting Braga,MID,66.7
3,GW 77,4,Extra MID,257.4,Cláudio Ramos,FC Porto,9.1,Samuel Dahl,SL Benfica,58.1,Geovany Quenda,Sporting Clube de Portugal,65.8,Deniz Gül,FC Porto,60.0,Maximiliano Araújo,Sporting Clube de Portugal,MID,64.4
4,GW 76,1,Extra MID,273.9,Lukáš Horníček,Sporting Braga,38.7,Víctor Gómez,Sporting Braga,63.0,João Moutinho,Sporting Braga,50.5,Pau Víctor,Sporting Braga,76.1,Vítor Carvalho,Sporting Braga,MID,45.6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
303,GW 2,4,Extra DEF,167.9,Arnas Voitinovičius,SL Benfica,0.0,António Silva,SL Benfica,45.4,Fredrik Aursnes,SL Benfica,43.6,Amine El Ouazzani,Sporting Braga,35.0,Samuel Dahl,SL Benfica,DEF,43.9
304,GW 1,1,Extra DEF,461.6,Lukáš Horníček,Sporting Braga,79.4,Gonçalo Inácio,Sporting Clube de Portugal,100.0,Victor Froholdt,FC Porto,99.6,Trincão,Sporting Clube de Portugal,82.6,Ricardo Mangas,Sporting Clube de Portugal,DEF,100.0
305,GW 1,2,Extra DEF,399.5,Rui Silva,Sporting Clube de Portugal,71.0,Zaidu,FC Porto,86.5,Diego Rodrigues,Sporting Braga,81.3,Ricardo Horta,Sporting Braga,75.9,Leonardo Lelo,Sporting Braga,DEF,84.8
306,GW 1,3,Extra DEF,378.5,Diogo Costa,FC Porto,64.8,Nehuen Pérez,FC Porto,83.7,Rafa,SL Benfica,79.0,Roger Fernandes,Sporting Braga,75.1,Paulo Oliveira,Sporting Braga,DEF,75.9


In [42]:
import pandas as pd
import os

# ============================================================
# SETTINGS
# ============================================================

OUTPUT_FOLDER = "sorare_outputs"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

rank1_file = os.path.join(OUTPUT_FOLDER, "rank_1_lineups_only_filtered.csv")
rank1_players_file = os.path.join(OUTPUT_FOLDER, "rank_1_most_common_players_filtered.csv")
analysis_excel_file = os.path.join(OUTPUT_FOLDER, "rank_1_lineups_analysis_filtered.xlsx")

# ============================================================
# 1. CREATE TEAM RANK 1 ONLY DATAFRAME
# ============================================================

df_rank1_raw = df_top_4_lineups[
    df_top_4_lineups["Team Rank"] == 1
].copy()

df_rank1_raw["GW Number"] = (
    df_rank1_raw["GW"]
    .astype(str)
    .str.replace("GW ", "", regex=False)
    .astype(int)
)

df_rank1_raw = df_rank1_raw.sort_values(
    "GW Number",
    ascending=False
).reset_index(drop=True)

print("=" * 100)
print("Raw Rank 1 lineups")
print("=" * 100)
print(f"Raw Rank 1 GWs: {len(df_rank1_raw)}")


# ============================================================
# 2. FILTER OUT GWS WHERE ANY SELECTED PLAYER HAS SCORE 0
# ============================================================

score_cols = [
    "GK Score",
    "DEF Score",
    "MID Score",
    "FWD Score",
    "Extra Score"
]

for col in score_cols:
    df_rank1_raw[col] = pd.to_numeric(
        df_rank1_raw[col],
        errors="coerce"
    ).fillna(0)

df_rank1 = df_rank1_raw[
    (df_rank1_raw["GK Score"] > 0) &
    (df_rank1_raw["DEF Score"] > 0) &
    (df_rank1_raw["MID Score"] > 0) &
    (df_rank1_raw["FWD Score"] > 0) &
    (df_rank1_raw["Extra Score"] > 0)
].copy()

df_removed_gws = df_rank1_raw[
    ~df_rank1_raw["GW"].isin(df_rank1["GW"])
].copy()

df_rank1 = df_rank1.reset_index(drop=True)

df_rank1.to_csv(rank1_file, index=False)

print("=" * 100)
print("Filtered Rank 1 lineups")
print("=" * 100)
print(f"Kept GWs: {len(df_rank1)}")
print(f"Removed GWs: {len(df_removed_gws)}")
print(f"Saved filtered Rank 1 lineups to: {rank1_file}")

print("=" * 100)
print("Removed GWs because at least one selected player had score 0")
print("=" * 100)

display(df_removed_gws[
    ["GW", "Formation", "Total Score"] + score_cols
])

print("=" * 100)
print("Filtered Rank 1 lineups used for analysis")
print("=" * 100)

display(df_rank1)


# ============================================================
# 3. FORMATION FREQUENCY
# ============================================================

df_formation_counts = (
    df_rank1["Formation"]
    .value_counts()
    .reset_index()
)

df_formation_counts.columns = ["Formation", "Count"]

df_formation_counts["Percentage"] = (
    df_formation_counts["Count"] / len(df_rank1) * 100
).round(2)

print("=" * 100)
print("Most common formations after filtering")
print("=" * 100)

display(df_formation_counts)


# ============================================================
# 4. SCORE NUMERICAL STATS
# ============================================================

score_stats = {
    "Raw Rank 1 GWs": len(df_rank1_raw),
    "Filtered Rank 1 GWs": len(df_rank1),
    "Removed GWs": len(df_removed_gws),
    "Highest Total Score": round(df_rank1["Total Score"].max(), 1),
    "Lowest Total Score": round(df_rank1["Total Score"].min(), 1),
    "Mean Total Score": round(df_rank1["Total Score"].mean(), 2),
    "Median Total Score": round(df_rank1["Total Score"].median(), 2),
    "Standard Deviation": round(df_rank1["Total Score"].std(), 2),
    "Q1": round(df_rank1["Total Score"].quantile(0.25), 2),
    "Q3": round(df_rank1["Total Score"].quantile(0.75), 2),
}

df_score_stats = pd.DataFrame(
    list(score_stats.items()),
    columns=["Metric", "Value"]
)

print("=" * 100)
print("Rank 1 total score stats after filtering")
print("=" * 100)

display(df_score_stats)


# ============================================================
# 5. BEST AND WORST GWS
# ============================================================

df_best_gws = df_rank1.sort_values(
    "Total Score",
    ascending=False
).head(10)

df_worst_gws = df_rank1.sort_values(
    "Total Score",
    ascending=True
).head(10)

print("=" * 100)
print("Top 10 best GWs after filtering")
print("=" * 100)

display(df_best_gws[
    ["GW", "Formation", "Total Score"]
])

print("=" * 100)
print("Top 10 worst GWs after filtering")
print("=" * 100)

display(df_worst_gws[
    ["GW", "Formation", "Total Score"]
])


# ============================================================
# 6. LONG PLAYER TABLE FROM FILTERED RANK 1
# ============================================================

player_role_columns = [
    ("GK", "GK Player", "GK Club", "GK Score"),
    ("DEF", "DEF Player", "DEF Club", "DEF Score"),
    ("MID", "MID Player", "MID Club", "MID Score"),
    ("FWD", "FWD Player", "FWD Club", "FWD Score"),
    ("Extra", "Extra Player", "Extra Club", "Extra Score")
]

player_rows = []

for _, row in df_rank1.iterrows():
    for role, player_col, club_col, score_col in player_role_columns:
        player_rows.append({
            "GW": row["GW"],
            "GW Number": row["GW Number"],
            "Role": role,
            "Player": row[player_col],
            "Club": row[club_col],
            "Score": row[score_col],
            "Formation": row["Formation"],
            "Total Team Score": row["Total Score"]
        })

df_rank1_players_long = pd.DataFrame(player_rows)


# ============================================================
# 7. MOST COMMON PLAYERS OVERALL
# ============================================================

df_most_common_players = (
    df_rank1_players_long
    .groupby(["Player", "Club"], as_index=False)
    .agg(
        Appearances=("GW", "count"),
        Average_Player_Score=("Score", "mean"),
        Median_Player_Score=("Score", "median"),
        Highest_Player_Score=("Score", "max"),
        Lowest_Player_Score=("Score", "min")
    )
)

df_most_common_players["Appearance Percentage"] = (
    df_most_common_players["Appearances"] / len(df_rank1) * 100
).round(2)

df_most_common_players = df_most_common_players.sort_values(
    ["Appearances", "Average_Player_Score"],
    ascending=[False, False]
).reset_index(drop=True)

for col in [
    "Average_Player_Score",
    "Median_Player_Score",
    "Highest_Player_Score",
    "Lowest_Player_Score"
]:
    df_most_common_players[col] = df_most_common_players[col].round(2)

df_most_common_players.to_csv(rank1_players_file, index=False)

print("=" * 100)
print("Most common players in filtered Rank 1 teams")
print("=" * 100)

display(df_most_common_players.head(30))


# ============================================================
# 8. MOST COMMON PLAYERS BY ROLE
# ============================================================

df_most_common_by_role = (
    df_rank1_players_long
    .groupby(["Role", "Player", "Club"], as_index=False)
    .agg(
        Appearances=("GW", "count"),
        Average_Score=("Score", "mean"),
        Highest_Score=("Score", "max"),
        Lowest_Score=("Score", "min")
    )
)

df_most_common_by_role["Appearance Percentage"] = (
    df_most_common_by_role["Appearances"] / len(df_rank1) * 100
).round(2)

for col in ["Average_Score", "Highest_Score", "Lowest_Score"]:
    df_most_common_by_role[col] = df_most_common_by_role[col].round(2)

df_most_common_by_role = df_most_common_by_role.sort_values(
    ["Role", "Appearances", "Average_Score"],
    ascending=[True, False, False]
).reset_index(drop=True)

print("=" * 100)
print("Most common players by role after filtering")
print("=" * 100)

display(df_most_common_by_role)


# ============================================================
# 9. CLUB REPRESENTATION
# ============================================================

df_club_representation = (
    df_rank1_players_long
    .groupby("Club", as_index=False)
    .agg(
        Total_Player_Slots=("Player", "count"),
        Average_Player_Score=("Score", "mean")
    )
)

df_club_representation["Slot Percentage"] = (
    df_club_representation["Total_Player_Slots"] / len(df_rank1_players_long) * 100
).round(2)

df_club_representation["Average_Player_Score"] = (
    df_club_representation["Average_Player_Score"].round(2)
)

df_club_representation = df_club_representation.sort_values(
    "Total_Player_Slots",
    ascending=False
).reset_index(drop=True)

print("=" * 100)
print("Club representation in filtered Rank 1 lineups")
print("=" * 100)

display(df_club_representation)


# ============================================================
# 10. SAVE ANALYSIS TABLES
# ============================================================

with pd.ExcelWriter(analysis_excel_file) as writer:
    df_rank1.to_excel(writer, sheet_name="Filtered Rank 1 Lineups", index=False)
    df_removed_gws.to_excel(writer, sheet_name="Removed GWs", index=False)
    df_formation_counts.to_excel(writer, sheet_name="Formations", index=False)
    df_score_stats.to_excel(writer, sheet_name="Score Stats", index=False)
    df_best_gws.to_excel(writer, sheet_name="Best GWs", index=False)
    df_worst_gws.to_excel(writer, sheet_name="Worst GWs", index=False)
    df_most_common_players.to_excel(writer, sheet_name="Common Players", index=False)
    df_most_common_by_role.to_excel(writer, sheet_name="Common By Role", index=False)
    df_club_representation.to_excel(writer, sheet_name="Club Representation", index=False)

print("=" * 100)
print("Filtered analysis files saved:")
print(rank1_file)
print(rank1_players_file)
print(analysis_excel_file)
print("=" * 100)

Raw Rank 1 lineups
Raw Rank 1 GWs: 77
Filtered Rank 1 lineups
Kept GWs: 59
Removed GWs: 18
Saved filtered Rank 1 lineups to: sorare_outputs\rank_1_lineups_only_filtered.csv
Removed GWs because at least one selected player had score 0


,GW,Formation,Total Score,GK Score,DEF Score,MID Score,FWD Score,Extra Score
11,GW 66,Extra DEF,307.1,0.0,93.4,73.0,67.3,73.4
13,GW 64,Extra DEF,269.7,0.0,81.4,70.2,46.8,71.3
19,GW 58,Extra DEF,0.0,0.0,0.0,0.0,0.0,0.0
25,GW 52,Extra DEF,85.6,0.0,0.0,0.0,85.6,0.0
27,GW 50,Extra DEF,0.0,0.0,0.0,0.0,0.0,0.0
33,GW 44,Extra DEF,0.0,0.0,0.0,0.0,0.0,0.0
34,GW 43,Extra MID,70.5,0.0,0.0,36.6,0.0,33.9
35,GW 42,Extra DEF,69.9,0.0,35.1,34.8,0.0,0.0
37,GW 40,Extra DEF,170.6,0.0,57.1,72.1,0.0,41.4
41,GW 36,Extra DEF,78.9,0.0,35.0,0.0,43.9,0.0


Filtered Rank 1 lineups used for analysis


,GW,Team Rank,Formation,Total Score,GK Player,GK Club,GK Score,DEF Player,DEF Club,DEF Score,...,MID Club,MID Score,FWD Player,FWD Club,FWD Score,Extra Player,Extra Club,Extra Position,Extra Score,GW Number
0,GW 77,1,Extra FWD,363.5,Lukáš Horníček,Sporting Braga,48.4,António Silva,SL Benfica,74.2,...,SL Benfica,79.1,Andreas Schjelderup,SL Benfica,83.6,Pau Víctor,Sporting Braga,FWD,78.2,77
1,GW 76,1,Extra MID,273.9,Lukáš Horníček,Sporting Braga,38.7,Víctor Gómez,Sporting Braga,63.0,...,Sporting Braga,50.5,Pau Víctor,Sporting Braga,76.1,Vítor Carvalho,Sporting Braga,MID,45.6,76
2,GW 75,1,Extra FWD,438.1,Diogo Costa,FC Porto,75.0,Jan Bednarek,FC Porto,100.0,...,Sporting Clube de Portugal,93.9,Luis Suárez,Sporting Clube de Portugal,86.3,Andreas Schjelderup,SL Benfica,FWD,82.9,75
3,GW 74,1,Extra MID,375.7,Rui Silva,Sporting Clube de Portugal,62.1,Víctor Gómez,Sporting Braga,84.2,...,Sporting Braga,84.4,Luis Suárez,Sporting Clube de Portugal,72.2,Maho Dorgeles,Sporting Braga,MID,72.8,74
4,GW 73,1,Extra MID,405.5,Rui Silva,Sporting Clube de Portugal,48.1,Zeno Debast,Sporting Clube de Portugal,85.0,...,SL Benfica,92.4,Oskar Pietuszewski,FC Porto,90.8,Leandro Barreiro,SL Benfica,MID,89.2,73
5,GW 72,1,Extra MID,340.9,Lukáš Horníček,Sporting Braga,66.6,Gustaf Lagerbielke,Sporting Braga,74.7,...,Sporting Braga,81.1,Pau Víctor,Sporting Braga,60.0,João Moutinho,Sporting Braga,MID,58.5,72
6,GW 71,1,Extra DEF,387.8,Anatolii Trubin,SL Benfica,82.7,Jan Bednarek,FC Porto,87.5,...,FC Porto,71.6,Ricardo Horta,Sporting Braga,73.4,Gonçalo Inácio,Sporting Clube de Portugal,DEF,72.6,71
7,GW 70,1,Extra FWD,345.0,Rui Silva,Sporting Clube de Portugal,67.8,Eduardo Quaresma,Sporting Clube de Portugal,68.2,...,Sporting Braga,65.6,Ricardo Horta,Sporting Braga,72.0,Pau Víctor,Sporting Braga,FWD,71.4,70
8,GW 69,1,Extra DEF,424.9,Anatolii Trubin,SL Benfica,79.6,Nicolás Otamendi,SL Benfica,90.5,...,FC Porto,88.6,Gianluca Prestianni,SL Benfica,88.1,Zaidu,FC Porto,DEF,78.1,69
9,GW 68,1,Extra MID,338.3,Lukáš Horníček,Sporting Braga,51.2,Zaidu,FC Porto,58.3,...,Sporting Braga,78.6,William,FC Porto,73.1,Gabri Veiga,FC Porto,MID,77.1,68


Most common formations after filtering


...

Most common players by role after filtering


,Role,Player,Club,Appearances,Average_Score,Highest_Score,Lowest_Score,Appearance Percentage
0,DEF,Jan Bednarek,FC Porto,7,87.99,100.0,68.1,11.86
1,DEF,Gonçalo Inácio,Sporting Clube de Portugal,6,90.00,100.0,74.1,10.17
2,DEF,Nicolás Otamendi,SL Benfica,6,88.00,100.0,69.3,10.17
3,DEF,Eduardo Quaresma,Sporting Clube de Portugal,4,92.05,100.0,68.2,6.78
4,DEF,Gustaf Lagerbielke,Sporting Braga,4,80.42,100.0,47.0,6.78
...,...,...,...,...,...,...,...,...
102,MID,Thiago Helguera,Sporting Braga,1,79.30,79.3,79.3,1.69
103,MID,Hidemasa Morita,Sporting Clube de Portugal,1,77.10,77.1,77.1,1.69
104,MID,Maho Dorgeles,Sporting Braga,1,72.90,72.9,72.9,1.69
105,MID,Leandro Barreiro,SL Benfica,1,70.40,70.4,70.4,1.69


Club representation in filtered Rank 1 lineups


,Club,Total_Player_Slots,Average_Player_Score,Slot Percentage
0,Sporting Braga,81,76.01,27.46
1,SL Benfica,80,78.62,27.12
2,Sporting Clube de Portugal,78,81.15,26.44
3,FC Porto,56,78.73,18.98


Filtered analysis files saved:
sorare_outputs\rank_1_lineups_only_filtered.csv
sorare_outputs\rank_1_most_common_players_filtered.csv
sorare_outputs\rank_1_lineups_analysis_filtered.xlsx


In [44]:
import pandas as pd
import numpy as np
import os

# ============================================================
# SETTINGS
# ============================================================

OUTPUT_FOLDER = "sorare_outputs"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

season_top_4_teams_file = os.path.join(
    OUTPUT_FOLDER,
    "season_consistency_top_4_teams_valid_gws_only.csv"
)

VALID_POSITIONS = ["GK", "DEF", "MID", "FWD"]
MIN_GAMES_PLAYED = 3

# ============================================================
# 1. GET VALID GWS FROM FILTERED RANK 1 LINEUPS
# ============================================================

# df_rank1 should be your filtered Rank 1 dataframe:
# only GWs where GK, DEF, MID, FWD and Extra all scored > 0.

valid_gw_cols = sorted(
    df_rank1["GW"].unique().tolist(),
    key=lambda x: int(str(x).replace("GW ", "")),
    reverse=True
)

print("=" * 100)
print("Valid GWs used for season-long analysis")
print("=" * 100)
print(f"Valid GWs: {len(valid_gw_cols)}")
print(valid_gw_cols)

# ============================================================
# 2. PREPARE PLAYER SEASON STATS USING ONLY VALID GWS
# ============================================================

df_season = df_players_gw.copy()

df_season = df_season[
    df_season["Position"].isin(VALID_POSITIONS)
].copy()

# Keep only GW columns that exist in df_players_gw
valid_gw_cols = [
    col for col in valid_gw_cols
    if col in df_season.columns
]

if not valid_gw_cols:
    raise Exception("No valid GW columns found in df_players_gw.")

for col in valid_gw_cols:
    df_season[col] = pd.to_numeric(
        df_season[col],
        errors="coerce"
    ).fillna(0)

score_matrix = df_season[valid_gw_cols]
played_matrix = score_matrix > 0

df_season["Valid GWs Analysed"] = len(valid_gw_cols)
df_season["Games Played"] = played_matrix.sum(axis=1)
df_season["Total Score"] = score_matrix.sum(axis=1).round(1)

# Availability only across valid GWs
df_season["Availability %"] = (
    df_season["Games Played"] / len(valid_gw_cols) * 100
).round(2)

# Average including zeros across valid GWs
df_season["Avg Score Including Zeros"] = score_matrix.mean(axis=1).round(2)

# Average only games where player scored
df_season["Avg Score When Played"] = (
    score_matrix.where(played_matrix)
    .mean(axis=1)
    .round(2)
)

df_season["Median Score When Played"] = (
    score_matrix.where(played_matrix)
    .median(axis=1)
    .round(2)
)

df_season["Std When Played"] = (
    score_matrix.where(played_matrix)
    .std(axis=1)
    .round(2)
)

df_season["Max Score"] = score_matrix.max(axis=1).round(1)

df_season["Min Positive Score"] = (
    score_matrix.where(played_matrix)
    .min(axis=1)
    .round(1)
)

df_season["Avg Score When Played"] = df_season["Avg Score When Played"].fillna(0)
df_season["Median Score When Played"] = df_season["Median Score When Played"].fillna(0)
df_season["Std When Played"] = df_season["Std When Played"].fillna(999)
df_season["Min Positive Score"] = df_season["Min Positive Score"].fillna(0)

# ============================================================
# 3. CONSISTENCY INDEX
# ============================================================

df_season["Consistency Index"] = (
    (df_season["Avg Score When Played"] * 0.35) +
    (df_season["Median Score When Played"] * 0.25) +
    (df_season["Avg Score Including Zeros"] * 0.25) +
    (df_season["Availability %"] * 0.10) -
    (df_season["Std When Played"].replace(999, 0) * 0.05)
).round(2)

df_candidates = df_season[
    df_season["Games Played"] >= MIN_GAMES_PLAYED
].copy()

df_candidates = df_candidates.sort_values(
    ["Consistency Index", "Avg Score Including Zeros", "Games Played", "Total Score"],
    ascending=[False, False, False, False]
).reset_index(drop=True)

print("=" * 100)
print("Top season-long candidates using valid GWs only")
print("=" * 100)

display(
    df_candidates[
        [
            "Club",
            "Player",
            "Position",
            "Valid GWs Analysed",
            "Games Played",
            "Availability %",
            "Total Score",
            "Avg Score Including Zeros",
            "Avg Score When Played",
            "Median Score When Played",
            "Std When Played",
            "Consistency Index"
        ]
    ].head(40)
)

# ============================================================
# 4. TEAM BUILDER
# ============================================================

def pick_best_available_player(available_df, position):
    candidates = available_df[
        available_df["Position"] == position
    ].copy()

    if candidates.empty:
        return None

    candidates = candidates.sort_values(
        ["Consistency Index", "Avg Score Including Zeros", "Games Played", "Total Score"],
        ascending=[False, False, False, False]
    )

    return candidates.iloc[0]


def build_season_team(available_df):
    selected = []

    for position in ["GK", "DEF", "MID", "FWD"]:
        player = pick_best_available_player(available_df, position)

        if player is None:
            return None

        selected.append(player)

        available_df = available_df[
            available_df["Player Slug"] != player["Player Slug"]
        ].copy()

    extra_pool = available_df[
        available_df["Position"].isin(["DEF", "MID", "FWD"])
    ].copy()

    if extra_pool.empty:
        return None

    extra_pool = extra_pool.sort_values(
        ["Consistency Index", "Avg Score Including Zeros", "Games Played", "Total Score"],
        ascending=[False, False, False, False]
    )

    extra_player = extra_pool.iloc[0]
    selected.append(extra_player)

    return pd.DataFrame(selected)

# ============================================================
# 5. BUILD TOP 4 SEASON-LONG TEAMS
# ============================================================

available_players = df_candidates.copy()
season_team_rows = []

for team_rank in range(1, 5):
    selected_team = build_season_team(available_players)

    if selected_team is None:
        print(f"Could not build Team {team_rank}")
        break

    base_players = selected_team.iloc[:4].copy()
    extra_player = selected_team.iloc[4].copy()

    row = {
        "Team Rank": team_rank,
        "Valid GWs Analysed": len(valid_gw_cols),
        "Team Consistency Index Sum": round(selected_team["Consistency Index"].sum(), 2),
        "Team Total Score Sum": round(selected_team["Total Score"].sum(), 1),
        "Team Avg Score Including Zeros Sum": round(selected_team["Avg Score Including Zeros"].sum(), 2),
        "Team Avg Score When Played Sum": round(selected_team["Avg Score When Played"].sum(), 2),
        "Team Games Played Sum": int(selected_team["Games Played"].sum())
    }

    for _, player in base_players.iterrows():
        pos = player["Position"]

        row[f"{pos} Player"] = player["Player"]
        row[f"{pos} Club"] = player["Club"]
        row[f"{pos} Games Played"] = int(player["Games Played"])
        row[f"{pos} Availability %"] = player["Availability %"]
        row[f"{pos} Total Score"] = player["Total Score"]
        row[f"{pos} Avg Including Zeros"] = player["Avg Score Including Zeros"]
        row[f"{pos} Avg When Played"] = player["Avg Score When Played"]
        row[f"{pos} Median When Played"] = player["Median Score When Played"]
        row[f"{pos} Consistency Index"] = player["Consistency Index"]

    row["Extra Player"] = extra_player["Player"]
    row["Extra Club"] = extra_player["Club"]
    row["Extra Position"] = extra_player["Position"]
    row["Extra Games Played"] = int(extra_player["Games Played"])
    row["Extra Availability %"] = extra_player["Availability %"]
    row["Extra Total Score"] = extra_player["Total Score"]
    row["Extra Avg Including Zeros"] = extra_player["Avg Score Including Zeros"]
    row["Extra Avg When Played"] = extra_player["Avg Score When Played"]
    row["Extra Median When Played"] = extra_player["Median Score When Played"]
    row["Extra Consistency Index"] = extra_player["Consistency Index"]

    season_team_rows.append(row)

    print("=" * 100)
    print(f"Season-long Team {team_rank}")
    print("=" * 100)
    print(f"Valid GWs Analysed: {len(valid_gw_cols)}")
    print(f"Team Consistency Index Sum: {row['Team Consistency Index Sum']}")
    print(f"Team Total Score Sum: {row['Team Total Score Sum']}")

    for _, player in selected_team.iterrows():
        label = "Extra" if player["Player Slug"] == extra_player["Player Slug"] else player["Position"]
        print(
            f"{label}: {player['Player']} | {player['Club']} | {player['Position']} | "
            f"Games: {int(player['Games Played'])} | "
            f"Availability: {player['Availability %']}% | "
            f"Avg+0: {player['Avg Score Including Zeros']} | "
            f"Avg Played: {player['Avg Score When Played']} | "
            f"Consistency: {player['Consistency Index']}"
        )

    selected_slugs = selected_team["Player Slug"].tolist()

    available_players = available_players[
        ~available_players["Player Slug"].isin(selected_slugs)
    ].copy()

# ============================================================
# 6. SAVE FINAL CSV
# ============================================================

df_season_top_4_teams = pd.DataFrame(season_team_rows)

df_season_top_4_teams.to_csv(season_top_4_teams_file, index=False)

print("=" * 100)
print("Season-long top 4 teams using valid GWs only created.")
print(f"Saved to: {season_top_4_teams_file}")
print("=" * 100)

display(df_season_top_4_teams)

Valid GWs used for season-long analysis
Valid GWs: 59
['GW 77', 'GW 76', 'GW 75', 'GW 74', 'GW 73', 'GW 72', 'GW 71', 'GW 70', 'GW 69', 'GW 68', 'GW 67', 'GW 65', 'GW 63', 'GW 62', 'GW 61', 'GW 60', 'GW 59', 'GW 57', 'GW 56', 'GW 55', 'GW 54', 'GW 53', 'GW 51', 'GW 49', 'GW 48', 'GW 47', 'GW 46', 'GW 45', 'GW 41', 'GW 39', 'GW 38', 'GW 37', 'GW 35', 'GW 34', 'GW 33', 'GW 31', 'GW 30', 'GW 27', 'GW 25', 'GW 24', 'GW 23', 'GW 21', 'GW 20', 'GW 18', 'GW 17', 'GW 15', 'GW 14', 'GW 13', 'GW 12', 'GW 11', 'GW 10', 'GW 9', 'GW 8', 'GW 7', 'GW 5', 'GW 4', 'GW 3', 'GW 2', 'GW 1']
Top season-long candidates using valid GWs only


GW Column,Club,Player,Position,Valid GWs Analysed,Games Played,Availability %,Total Score,Avg Score Including Zeros,Avg Score When Played,Median Score When Played,Std When Played,Consistency Index
0,Sporting Clube de Portugal,Luis Suárez,FWD,59,45,76.27,2714.4,46.01,60.32,60.00,20.17,54.23
1,Sporting Braga,Ricardo Horta,FWD,59,40,67.80,2496.7,42.32,62.42,63.35,21.15,53.99
2,Sporting Clube de Portugal,Trincão,FWD,59,47,79.66,2806.9,47.57,59.72,55.10,19.51,53.56
3,FC Porto,Jan Bednarek,DEF,59,44,74.58,2574.1,43.63,58.50,59.85,20.57,52.77
4,SL Benfica,Nicolás Otamendi,DEF,59,43,72.88,2581.5,43.75,60.03,57.40,21.19,52.53
5,Sporting Clube de Portugal,Gonçalo Inácio,DEF,59,40,67.80,2477.6,41.99,61.94,56.90,19.48,52.21
6,FC Porto,Jakub Kiwior,DEF,59,36,61.02,2201.8,37.32,61.16,61.45,18.65,51.27
7,FC Porto,Diogo Costa,GK,59,48,81.36,2598.8,44.05,54.14,56.20,19.39,51.18
8,Sporting Braga,Víctor Gómez,DEF,59,44,74.58,2410.1,40.85,54.78,56.25,15.47,50.13
9,Sporting Clube de Portugal,Morten Hjulmand,MID,59,38,64.41,2206.8,37.40,58.07,56.70,17.53,49.41


Season-long Team 1
Valid GWs Analysed: 59
Team Consistency Index Sum: 261.58
Team Total Score Sum: 12590.8
GK: Diogo Costa | FC Porto | GK | Games: 48 | Availability: 81.36% | Avg+0: 44.05 | Avg Played: 54.14 | Consistency: 51.18
DEF: Jan Bednarek | FC Porto | DEF | Games: 44 | Availability: 74.58% | Avg+0: 43.63 | Avg Played: 58.5 | Consistency: 52.77
MID: Morten Hjulmand | Sporting Clube de Portugal | MID | Games: 38 | Availability: 64.41% | Avg+0: 37.4 | Avg Played: 58.07 | Consistency: 49.41
FWD: Luis Suárez | Sporting Clube de Portugal | FWD | Games: 45 | Availability: 76.27% | Avg+0: 46.01 | Avg Played: 60.32 | Consistency: 54.23
Extra: Ricardo Horta | Sporting Braga | FWD | Games: 40 | Availability: 67.8% | Avg+0: 42.32 | Avg Played: 62.42 | Consistency: 53.99
Season-long Team 2
Valid GWs Analysed: 59
Team Consistency Index Sum: 255.54
Team Total Score Sum: 12667.0
GK: Lukáš Horníček | Sporting Braga | GK | Games: 46 | Availability: 77.97% | Avg+0: 42.37 | Avg Played: 54.34 | Co

,Team Rank,Valid GWs Analysed,Team Consistency Index Sum,Team Total Score Sum,Team Avg Score Including Zeros Sum,Team Avg Score When Played Sum,Team Games Played Sum,GK Player,GK Club,GK Games Played,...,Extra Player,Extra Club,Extra Position,Extra Games Played,Extra Availability %,Extra Total Score,Extra Avg Including Zeros,Extra Avg When Played,Extra Median When Played,Extra Consistency Index
0,1,59,261.58,12590.8,213.41,293.45,215,Diogo Costa,FC Porto,48,...,Ricardo Horta,Sporting Braga,FWD,40,67.80,2496.7,42.32,62.42,63.35,53.99
1,2,59,255.54,12667.0,214.68,292.16,217,Lukáš Horníček,Sporting Braga,46,...,Gonçalo Inácio,Sporting Clube de Portugal,DEF,40,67.80,2477.6,41.99,61.94,56.90,52.21
2,3,59,244.29,11723.8,198.70,275.34,214,Anatolii Trubin,SL Benfica,47,...,Víctor Gómez,Sporting Braga,DEF,44,74.58,2410.1,40.85,54.78,56.25,50.13
3,4,59,233.44,11183.2,189.55,265.21,211,Rui Silva,Sporting Clube de Portugal,42,...,Gustaf Lagerbielke,Sporting Braga,DEF,43,72.88,2214.1,37.53,51.49,49.70,46.11


## Post-extraction analysis

These cells read from the extracted CSV, so you can run them after extraction finishes or after restarting the notebook.

In [ ]:
import os
import pandas as pd
import numpy as np

OUTPUT_FOLDER = "sorare_outputs"
SCORES_FILE = os.path.join(OUTPUT_FOLDER, "all_players_scores_final.csv")

if not os.path.exists(SCORES_FILE):
    raise FileNotFoundError(f"Run the extraction first. Missing: {SCORES_FILE}")

df_scores_analysis = pd.read_csv(SCORES_FILE)
df_scores_analysis["Game Date"] = pd.to_datetime(df_scores_analysis["Game Date"], utc=True, errors="coerce")
df_scores_analysis["Score"] = pd.to_numeric(df_scores_analysis["Score"], errors="coerce")
df_scores_analysis["Sorare GW"] = pd.to_numeric(df_scores_analysis["Sorare GW"], errors="coerce")

print("Shape:", df_scores_analysis.shape)
print("Date range:", df_scores_analysis["Game Date"].min(), "to", df_scores_analysis["Game Date"].max())
print("Players:", df_scores_analysis["Player Slug"].nunique())
print("Clubs:", df_scores_analysis["Club Slug"].nunique())
display(df_scores_analysis.head())

In [ ]:
df_scores_analysis = df_scores_analysis.sort_values(["Player Slug", "Game Date"]).copy()

player_group = df_scores_analysis.groupby("Player Slug", group_keys=False)
# Shift before rolling to avoid lookahead bias: current row only uses prior matches.
df_scores_analysis["rolling_5_avg_score"] = player_group["Score"].apply(lambda s: s.shift(1).rolling(5, min_periods=1).mean())
df_scores_analysis["rolling_15_avg_score"] = player_group["Score"].apply(lambda s: s.shift(1).rolling(15, min_periods=1).mean())
df_scores_analysis["form_delta"] = df_scores_analysis["rolling_5_avg_score"] - df_scores_analysis["rolling_15_avg_score"]

player_rankings = (
    df_scores_analysis.groupby(["Club", "Club Slug", "Player", "Player Slug", "Position"], dropna=False)
    .agg(
        games=("Score", "count"),
        avg_score=("Score", "mean"),
        median_score=("Score", "median"),
        score_volatility=("Score", "std"),
        score_floor=("Score", "min"),
        score_ceiling=("Score", "max"),
        recent_form=("rolling_5_avg_score", "last"),
        longer_form=("rolling_15_avg_score", "last"),
        form_delta=("form_delta", "last"),
    )
    .reset_index()
)
player_rankings["consistency_index"] = player_rankings["avg_score"] / (1 + player_rankings["score_volatility"].fillna(0))
player_rankings["risk_reward_index"] = player_rankings["score_ceiling"] - player_rankings["score_floor"]
player_rankings = player_rankings.sort_values(["avg_score", "games"], ascending=[False, False])

rankings_file = os.path.join(OUTPUT_FOLDER, "player_rankings.csv")
player_rankings.to_csv(rankings_file, index=False)
print(f"Saved rankings to: {rankings_file}")
display(player_rankings.head(30))

In [ ]:
MIN_GAMES = 5

most_consistent = player_rankings[player_rankings["games"] >= MIN_GAMES].sort_values("consistency_index", ascending=False)
highest_average = player_rankings[player_rankings["games"] >= MIN_GAMES].sort_values("avg_score", ascending=False)
highest_ceiling = player_rankings[player_rankings["games"] >= MIN_GAMES].sort_values("score_ceiling", ascending=False)
improving = player_rankings[player_rankings["games"] >= MIN_GAMES].sort_values("form_delta", ascending=False)
declining = player_rankings[player_rankings["games"] >= MIN_GAMES].sort_values("form_delta", ascending=True)
high_risk_high_reward = player_rankings[player_rankings["games"] >= MIN_GAMES].sort_values("risk_reward_index", ascending=False)

print("Most consistent")
display(most_consistent.head(15))
print("Highest average")
display(highest_average.head(15))
print("Highest ceiling")
display(highest_ceiling.head(15))
print("Improving")
display(improving.head(15))
print("Declining")
display(declining.head(15))
print("High risk / high reward")
display(high_risk_high_reward.head(15))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

charts_dir = os.path.join("outputs", "charts")
os.makedirs(charts_dir, exist_ok=True)
sns.set_theme(style="whitegrid")

plt.figure(figsize=(10, 5))
sns.histplot(df_scores_analysis["Score"].dropna(), bins=30, kde=True)
plt.title("Sorare Score Distribution")
plt.tight_layout()
plt.savefig(os.path.join(charts_dir, "score_distribution_histogram.png"), dpi=150)
plt.show()

plt.figure(figsize=(10, 5))
sns.boxplot(data=df_scores_analysis, x="Position", y="Score")
plt.title("Scores by Position")
plt.tight_layout()
plt.savefig(os.path.join(charts_dir, "scores_by_position_boxplot.png"), dpi=150)
plt.show()

heatmap_data = df_scores_analysis.pivot_table(index="Position", columns="Sorare GW", values="Score", aggfunc="mean")
plt.figure(figsize=(14, 4))
sns.heatmap(heatmap_data, cmap="viridis")
plt.title("Average Score by Position and Gameweek")
plt.tight_layout()
plt.savefig(os.path.join(charts_dir, "avg_score_position_gameweek_heatmap.png"), dpi=150)
plt.show()

print(f"Charts saved to: {charts_dir}")

## Season-aware club matching

Optional: create `data/club_season_squads.csv` with one row per player-club-season membership. The code below assigns each score to a club only when the player belonged to one of your tracked clubs during that score date. If no membership matches, the matched club stays null.

In [ ]:
import os
import pandas as pd

SQUAD_MEMBERSHIP_FILE = os.path.join("data", "club_season_squads.csv")
MATCHED_LONG_FILE = os.path.join("sorare_outputs", "club_player_gw_matched_scores.csv")
MATCHED_WIDE_FILE = os.path.join("sorare_outputs", "club_player_gw_matched_scores_wide.csv")

required_squad_columns = [
    "Club",
    "Club Slug",
    "Player Slug",
    "Season Start",
    "Season End",
]

if not os.path.exists(SQUAD_MEMBERSHIP_FILE):
    print(f"Create {SQUAD_MEMBERSHIP_FILE} to enable season-aware club matching.")
    print("Required columns:", required_squad_columns)
else:
    df_squads = pd.read_csv(SQUAD_MEMBERSHIP_FILE)
    missing = [col for col in required_squad_columns if col not in df_squads.columns]
    if missing:
        raise ValueError(f"Squad file is missing columns: {missing}")

    df_squads["Season Start"] = pd.to_datetime(df_squads["Season Start"], utc=True, errors="coerce")
    df_squads["Season End"] = pd.to_datetime(df_squads["Season End"], utc=True, errors="coerce")

    source_scores = df_scores.copy() if "df_scores" in globals() else pd.read_csv(final_scores_file)
    source_scores["Game Date"] = pd.to_datetime(source_scores["Game Date"], utc=True, errors="coerce")
    source_scores["_score_row_id"] = range(len(source_scores))

    matches = source_scores.merge(
        df_squads[["Club", "Club Slug", "Player Slug", "Season Start", "Season End"]].rename(
            columns={
                "Club": "Matched Club",
                "Club Slug": "Matched Club Slug",
            }
        ),
        on="Player Slug",
        how="left"
    )

    matches = matches[
        (matches["Game Date"] >= matches["Season Start"]) &
        (matches["Game Date"] <= matches["Season End"])
    ].copy()

    matches = (
        matches
        .sort_values(["_score_row_id", "Season Start"])
        .drop_duplicates("_score_row_id", keep="last")
    )

    match_cols = matches[[
        "_score_row_id",
        "Matched Club",
        "Matched Club Slug",
        "Season Start",
        "Season End",
    ]].rename(columns={
        "Season Start": "Matched Season Start",
        "Season End": "Matched Season End",
    })

    df_scores_matched = source_scores.merge(match_cols, on="_score_row_id", how="left").drop(columns=["_score_row_id"])
    df_scores_matched["Matched In Dataset Club"] = df_scores_matched["Matched Club Slug"].notna()
    df_scores_matched.to_csv(MATCHED_LONG_FILE, index=False)

    df_for_wide = df_scores_matched[df_scores_matched["Matched In Dataset Club"]].copy()
    df_for_wide["GW Column"] = "GW " + df_for_wide["Sorare GW"].astype(int).astype(str)

    if df_for_wide.empty:
        df_matched_wide = pd.DataFrame()
    else:
        df_matched_wide = df_for_wide.pivot_table(
            index=["Matched Club", "Matched Club Slug", "Player", "Player Slug", "Position"],
            columns="GW Column",
            values="Score",
            aggfunc="first"
        ).reset_index()

        gw_cols = sorted(
            [col for col in df_matched_wide.columns if str(col).startswith("GW ")],
            key=lambda x: int(str(x).replace("GW ", "")),
            reverse=True
        )
        id_cols = ["Matched Club", "Matched Club Slug", "Player", "Player Slug", "Position"]
        df_matched_wide = df_matched_wide[id_cols + gw_cols]

    df_matched_wide.to_csv(MATCHED_WIDE_FILE, index=False)

    print(f"Saved matched long table to: {MATCHED_LONG_FILE}")
    print(f"Saved matched wide table to: {MATCHED_WIDE_FILE}")
    print("Matched rows:", int(df_scores_matched["Matched In Dataset Club"].sum()))
    print("Unmatched rows with null current club in that GW:", int((~df_scores_matched["Matched In Dataset Club"]).sum()))
    display(df_scores_matched.head(20))

## Build club x player x GW from Transfermarkt matchday squads

This uses the public Transfermarkt dataset to build matchday squad rows for the tracked clubs, then matches Sorare score rows by player name and match date. Unmatched rows keep club as null.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from build_squad_memberships import build_matchday_squads, match_sorare_scores_to_clubs

scores_path = PROJECT_ROOT / "sorare_outputs" / "all_players_scores_final.csv"
club_map_path = PROJECT_ROOT / "config" / "club_name_map.csv"
squads_output = PROJECT_ROOT / "data" / "club_matchday_squads.csv"
matched_output = PROJECT_ROOT / "sorare_outputs" / "club_player_gw_matched_scores.csv"

if not scores_path.exists():
    raise FileNotFoundError(f"Run extraction first. Missing: {scores_path}")

build_matchday_squads(
    club_map_path=club_map_path,
    start_date=SEASON_START_DATE,
    output_path=squads_output,
)

df_scores_matched = match_sorare_scores_to_clubs(
    scores_path=scores_path,
    squads_path=squads_output,
    output_path=matched_output,
    tolerance_days=1,
)

display(df_scores_matched.head(30))